# Lesson 03 - Modele de Proiectare Agentice

În această lecție, explorăm trei modele fundamentale de proiectare pentru construirea agenților AI eficienți:

1. **Instrucțiuni Clare pentru Agent** — Crearea de solicitări precise, care definesc rolul și ghidează comportamentul agentului  
2. **Ieșire Structurată cu Modele Pydantic** — Asigurarea că agenții returnează date previzibile și validate  
3. **Agenți cu Responsabilitate Unică** — Proiectarea de agenți concentrați care fac bine o singură sarcină

Vom aplica fiecare model într-un scenariu de **recomandare a destinațiilor de călătorie**, construind progresiv un sistem care poate sugera destinații, verifica disponibilitatea și gestiona logistica.


## Configurare


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Modelul 1: Instrucțiuni clare pentru agent

Cel mai eficient model este și cel mai simplu: scrierea unor instrucțiuni clare și detaliate pentru agentul tău.

Instrucțiunile bune definesc:
- **Cine** este agentul (persoană și ton)
- **Ce** ar trebui să facă (responsabilități pas cu pas)
- **Cum** ar trebui să se comporte (restricții și stil)

Mai jos, creăm un agent concierge de călătorie cu instrucțiuni explicite care modelează fiecare răspuns pe care îl produce.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Ieșire Structurată cu Modele Pydantic

Textul liber este util pentru conversație, dar sistemele ulterioare au nevoie de date structurate.
Prin asocierea **modelor Pydantic** cu o **funcție de unealtă**, putem:

- Defini un schema exactă pentru ieșirea agentului
- Valida răspunsurile automat
- Integra rezultatele agentului în logica aplicației în mod fiabil

De asemenea, introducem o unealtă care returnează detalii despre destinație astfel încât agentul să își bazeze recomandările pe date reale.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Modelul 3: Agenți cu Responsabilitate Unică

Sarcinile complexe beneficiază de împărțirea muncii între mai mulți agenți concentrați, fiecare având o singură responsabilitate:

- Un **Expert în Destinații** care cunoaște locurile și disponibilitatea
- Un **Planificator Logistic** care se ocupă de zboruri, hoteluri și itinerarii

Aceasta reflectă principiul ingineriei software de *separare a preocupărilor* — fiecare agent este mai ușor de testat, întreținut și îmbunătățit independent.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Rezumat

În această lecție am aplicat trei modele de design agentic într-un scenariu de recomandare pentru călătorii:

| Model | Idee cheie | Beneficiu |
|---|---|---|
| **Instrucțiuni Clare** | Definirea personajului, responsabilităților și limitărilor de la început | Comportament consistent al agentului, în conformitate cu brandul |
| **Output Structurat** | Folosirea modelelor Pydantic ca format al răspunsului | Rezultate validate, ușor de citit de mașini |
| **Responsabilitate Unică** | Fiecare agent primește un singur rol bine definit | Mai ușor de testat, întreținut și compus |

Aceste modele se compun natural — poți combina instrucțiuni clare cu output structurat într-un agent cu responsabilitate unică pentru a crea sisteme robuste, gata de producție.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
